In [ ]:
"""
03_channel_metadata.py
======================
Enrich chart dataset with YouTube metadata
using the official YouTube Data API v3.

INPUT  : india_youtube_weekly_charts_2021_2026.csv
OUTPUT : india_youtube_enriched.csv
"""

from __future__ import annotations

import logging
import time
from pathlib import Path
from typing import Iterator

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ── Config ────────────────────────────────────────────────────────────────────

API_KEY    = ""
BASE_URL   = "https://www.googleapis.com/youtube/v3/videos"
INPUT_FILE = Path("india_youtube_weekly_charts_2021_2026.csv")
OUTPUT_FILE = Path("india_youtube_enriched.csv")

BATCH_SIZE      = 50       # YouTube API hard limit
REQUEST_DELAY   = 0.2      # seconds between batches
MAX_RETRIES     = 3        # retries on transient failures
BACKOFF_FACTOR  = 1.5      # exponential backoff multiplier


# ── Logging ───────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ── HTTP session with retry ───────────────────────────────────────────────────

def build_session() -> requests.Session:
    """Return a session with automatic retry on transient HTTP errors."""
    session = requests.Session()
    retry = Retry(
        total=MAX_RETRIES,
        backoff_factor=BACKOFF_FACTOR,
        status_forcelist={429, 500, 502, 503, 504},
        allowed_methods={"GET"},
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


# ── Helpers ───────────────────────────────────────────────────────────────────

def batched(items: list, size: int) -> Iterator[list]:
    """Yield successive fixed-size slices of *items*."""
    for i in range(0, len(items), size):
        yield items[i : i + size]


def parse_item(item: dict) -> dict:
    """Flatten a single YouTube API video resource into a plain dict."""
    snippet    = item.get("snippet", {})
    statistics = item.get("statistics", {})
    content    = item.get("contentDetails", {})

    return {
        "video_id":             item.get("id"),
        "channel_id":           snippet.get("channelId"),
        "channel_title":        snippet.get("channelTitle"),
        "video_title":          snippet.get("title"),
        "description":          snippet.get("description"),
        "published_at":         snippet.get("publishedAt"),
        "tags":                 snippet.get("tags"),          # list or None
        "category_id":          snippet.get("categoryId"),
        "default_language":     snippet.get("defaultLanguage"),
        "duration":             content.get("duration"),     # ISO 8601 e.g. PT3M45S
        "youtube_total_views":  _to_int(statistics.get("viewCount")),
        "youtube_like_count":   _to_int(statistics.get("likeCount")),
        "youtube_comment_count":_to_int(statistics.get("commentCount")),
    }


def _to_int(value: str | None) -> int | None:
    """Safely cast a string count to int."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


# ── Core fetch ────────────────────────────────────────────────────────────────

def fetch_metadata(
    video_ids: list[str],
    session: requests.Session,
) -> list[dict]:
    """
    Fetch video metadata for all *video_ids* in batches of up to 50.
    Returns a list of flattened metadata dicts.
    """
    rows: list[dict] = []
    total_batches = -(-len(video_ids) // BATCH_SIZE)  # ceiling division

    for batch_num, batch in enumerate(batched(video_ids, BATCH_SIZE), start=1):
        log.info("Batch %d / %d  (%d IDs)", batch_num, total_batches, len(batch))

        try:
            response = session.get(
                BASE_URL,
                params={
                    "part": "snippet,statistics,contentDetails",
                    "id":   ",".join(batch),
                    "key":  API_KEY,
                },
                timeout=10,
            )
            response.raise_for_status()
        except requests.exceptions.HTTPError as exc:
            log.error("HTTP error on batch %d: %s", batch_num, exc)
            continue
        except requests.exceptions.RequestException as exc:
            log.error("Request failed on batch %d: %s", batch_num, exc)
            continue

        data  = response.json()
        items = data.get("items", [])

        # Warn if the API returned fewer items than expected
        if len(items) < len(batch):
            returned_ids = {item["id"] for item in items}
            missing = set(batch) - returned_ids
            log.warning(
                "Batch %d: %d IDs not returned by API — possibly deleted/private: %s",
                batch_num, len(missing), missing,
            )

        rows.extend(parse_item(item) for item in items)
        time.sleep(REQUEST_DELAY)

    log.info("Fetched metadata for %d videos total", len(rows))
    return rows


# ── Main ──────────────────────────────────────────────────────────────────────

def main() -> None:
    # --- Load chart data ---
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

    log.info("Loading chart data from %s", INPUT_FILE)
    df = pd.read_csv(INPUT_FILE)
    log.info("Shape: %s", df.shape)

    if "video_id" not in df.columns:
        raise ValueError("Expected a 'video_id' column in the input CSV.")

    # --- Deduplicate IDs ---
    video_ids = df["video_id"].dropna().unique().tolist()
    log.info("Unique video IDs to fetch: %d", len(video_ids))

    # --- Fetch ---
    session = build_session()
    rows    = fetch_metadata(video_ids, session)

    if not rows:
        log.error("No metadata returned — check your API key and video IDs.")
        return

    # --- Build metadata dataframe ---
    meta_df = pd.DataFrame(rows)

    # Normalise dtypes
    meta_df["published_at"] = pd.to_datetime(meta_df["published_at"], errors="coerce", utc=True)
    meta_df["category_id"]  = pd.to_numeric(meta_df["category_id"], errors="coerce").astype("Int64")

    log.info("Metadata shape: %s", meta_df.shape)

    # --- Merge ---
    merged_df = df.merge(meta_df, on="video_id", how="left")
    log.info("Merged shape: %s", merged_df.shape)

    # Sanity check: rows should not multiply
    if len(merged_df) != len(df):
        log.warning(
            "Row count changed after merge (%d → %d). "
            "Check for duplicate video_ids in metadata.",
            len(df), len(merged_df),
        )

    # --- Save ---
    merged_df.to_csv(OUTPUT_FILE, index=False)
    log.info("Saved → %s", OUTPUT_FILE)


if __name__ == "__main__":
    main()

01:33:19  INFO      Loading chart data from india_youtube_weekly_charts_2021_2026.csv
01:33:19  INFO      Shape: (28000, 9)
01:33:19  INFO      Unique video IDs to fetch: 3337
01:33:19  INFO      Batch 1 / 67  (50 IDs)
01:33:20  INFO      Batch 2 / 67  (50 IDs)
01:33:21  INFO      Batch 3 / 67  (50 IDs)
01:33:21  INFO      Batch 4 / 67  (50 IDs)
01:33:22  INFO      Batch 5 / 67  (50 IDs)
01:33:22  INFO      Batch 6 / 67  (50 IDs)
01:33:23  INFO      Batch 7 / 67  (50 IDs)
01:33:23  INFO      Batch 8 / 67  (50 IDs)
01:33:24  INFO      Batch 9 / 67  (50 IDs)
01:33:24  INFO      Batch 10 / 67  (50 IDs)
01:33:25  INFO      Batch 11 / 67  (50 IDs)
01:33:25  INFO      Batch 12 / 67  (50 IDs)
01:33:26  INFO      Batch 13 / 67  (50 IDs)
01:33:26  INFO      Batch 14 / 67  (50 IDs)
01:33:27  INFO      Batch 15 / 67  (50 IDs)
01:33:28  INFO      Batch 16 / 67  (50 IDs)
01:33:28  INFO      Batch 17 / 67  (50 IDs)
01:33:29  INFO      Batch 18 / 67  (50 IDs)
01:33:29  INFO      Batch 19 / 67  (50 ID